# 4.2.1 Re-clustering Immune fraction

**Inés Sentís**

Date of execution

In [ ]:
Sys.Date()

## Libraries

In [ ]:
suppressPackageStartupMessages({
    library(Seurat)
    library(Signac)
    library(tidyverse)
    library(harmony)
    library(scater)
    library(dittoSeq)
    library(scran)
    library(here)
    library(glue)})

## Parameters

In [ ]:
source(here("utils/bin.R"))
source(here("utils/paths.R"))
source(here("AVGUSTINOVA_02_to_30/utils/style.R"))
source(here("AVGUSTINOVA_02_to_30/utils/variables.R"))

"{int}/{robj_dir}" %>%
    glue::glue() %>%
    here::here() %>%
    dir.create(path = .,
               showWarnings = FALSE,
               recursive = TRUE)
set.seed(0)

In [ ]:
combined_obj <- readRDS(here(glue("{int}/{robj_dir}/4.2.0-immune_fraction.rds")))

### High variable features

Remove dimorfic genes according to [this paper](https://www.nature.com/articles/ncomms10717/figures/1).

In [ ]:
genes_dmf <- c("XACT", "HTR2C",
               "XIST", "DDX3Y","USP9Y","NLGN4Y", "KDM5D", "EIF1AY", "UBA1")
mt_genes <- grep("^MT-", rownames(combined_obj[["RNA"]]), value = TRUE)
genes_out <- c(genes_dmf, top_c, mt_genes) 

In [ ]:
list_objects <- SplitObject(combined_obj, split.by = "patient")

In [ ]:
find_hvgs <-  function(obj) {
    
    pat <- unique(obj$patient)
    
    # remove genes from matrix before modulating variance and mean
    DefaultAssay(obj) <- "RNA"
    df_meta <- obj@meta.data
    counts <- obj[['RNA']]@counts
    counts <- counts[-(which(rownames(counts) %in% genes_out)),]
    obj2 <- CreateSeuratObject(counts = counts, meta.data = df_meta)
    
    # normalize again
    obj2 <- obj2 %>% 
            NormalizeData(
                normalization.method = "LogNormalize",
                scale.factor = 1e4) 
    
    # compute the variance and mean of the log-expression values 
    sce <- as.SingleCellExperiment(obj2)
    gene_var <- modelGeneVar(sce)
    
    tops <- gene_var %>% 
        as.data.frame() %>% 
        arrange(desc(total)) %>% 
        head(n=20)

    plt <- gene_var %>% 
      as.data.frame() %>% 
      ggplot(aes(mean, total)) +
      geom_point() +
      labs(x = "Mean of log-expression", y = "Variance of log-expression")+
      geom_text(data=tops, aes(mean,total,label=rownames(tops))) + 
      ggtitle(pat) + 
      theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold"))

    hvgs <- getTopHVGs(gene_var,fdr.threshold = 0.05)
    return(list(plot = plt, hvgs_pat = hvgs))
}

In [ ]:
modelling <- lapply(list_objects, find_hvgs)

In [ ]:
list_plots <- lapply(modelling, function(el) {
    el$plot
})

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15)
cowplot::plot_grid(plotlist = list_plots,
                   align = "hv",
                   axis = "trbl",
                   ncol=4)

In [ ]:
list_hvg <- lapply(modelling, function(el) {
    hvg <- el$hvgs_pat
    print(length(hvg))
    hvg
})

In [ ]:
hvg_common <- find_perc(list_hvg, perc = 2/15)
length(hvg_common)

### Normalization and linear dimensional reduction

In [ ]:
VariableFeatures(combined_obj) <- hvg_common
 
combined_obj <- combined_obj %>% 
  NormalizeData(normalization.method = "LogNormalize",
                scale.factor = 1e4) %>%
  ScaleData(features=hvg_common) %>% 
  RunPCA(features=hvg_common) 

## Normalization and Embedding

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 8)
ElbowPlot(combined_obj, ndims=50)

In [ ]:
rna_comp = 1:25

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 6)
VizDimLoadings(combined_obj, dims = 1:5, reduction = "pca", ncol=5)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)
new_markers = c("PTPRC", "DOCK4", "MSR1", "CTSB","TOX","SKAP1", "CD2", "CD247")
FeaturePlot(combined_obj, reduction = "pca", features = new_markers,
           ncol=4, order=T)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)
old_markers = c("PTPRC", "CD3D", "CD3E", "CD4","CD8A", "CD8B", "CD68", "CD14")
FeaturePlot(combined_obj, reduction = "pca", 
            features = old_markers,
           ncol=4, order=T)

### Harmony

In [ ]:
# make a copy of the PCA reduction
combined_obj@reductions$pca_filt <- combined_obj@reductions$pca

# Filter to the desired number of components according to elbow
combined_obj@reductions$pca_filt@cell.embeddings <- combined_obj@reductions$pca_filt@cell.embeddings[, rna_comp]

combined_obj@reductions$pca_filt@feature.loadings <- combined_obj@reductions$pca_filt@feature.loadings[, rna_comp]


combined_obj <- RunHarmony(
  object = combined_obj,
  reduction = "pca_filt",
  dims = rna_comp,
  group.by.vars = c("patient"),
  assay.use = "RNA",
  max_iter = 15,
  project.dim = T,
  reduction.save = "harmony_pca"
)

In [ ]:
combined_obj <- RunUMAP(
  combined_obj,
  dims = rna_comp,
  reduction = "harmony_pca",
  reduction.name = "umap.rna.harmony",
  reduction.key = "RNAhar_",
  assay = "RNA")

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 8)
DimPlot(
   combined_obj,
   reduction = "umap.rna.harmony",
   group.by = "patient",
   pt.size = 0.05,
   cols= colors_patients,
) + ggtitle('RNA UMAP\nby patient') + 
    theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold")) 

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)
FeaturePlot(combined_obj, reduction = "umap.rna.harmony",features = new_markers, ncol=4, order=T)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)
FeaturePlot(combined_obj, reduction = "umap.rna.harmony",features = old_markers, ncol=4, order=T)

## Clustering

In [ ]:
# Find the K-nearest neighbors
combined_obj <- FindNeighbors(combined_obj, reduction = "harmony_pca", 
                            dims = rna_comp)

In [ ]:
combined_obj <- FindClusters(combined_obj, resolution = 0.20, graph.name = "RNA_snn")

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 8)
DimPlot(combined_obj, reduction = "umap.rna.harmony", label=T) +
    ggtitle('Clustering') + 
    theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold"))

In [ ]:
combined_obj <- FindSubCluster(
  object=combined_obj,
  cluster="6",
  graph.name="RNA_snn",
  subcluster.name = "subclustering",
  resolution = 0.2,
  algorithm = 1
)

DimPlot(combined_obj, reduction = "umap.rna.harmony", label=T,
                  group.by=glue::glue("subclustering")) +
    labs(title=glue::glue("Subclustering"))

In [ ]:
combined_obj <- SetIdent(combined_obj, value=combined_obj@meta.data[["subclustering"]])

In [ ]:
rna.markers <- FindAllMarkers(combined_obj, 
                              logfc.threshold = 0.1, only.pos = TRUE)

In [ ]:
rna.markers <- rna.markers %>% 
  dplyr::filter(p_val_adj < 0.05) %>% 
  arrange(desc(avg_log2FC))

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 5, warn=-1,verbose = FALSE)
dp <- dittoSeq::dittoDotPlot(subset(combined_obj,ident=c("0","2","3","6_1")), 
                             vars=c("CD14","S100A8","S100A9","FCGR3A","CD300E","VCAN",
                                    "CD68","SPP1","CD163","MRC1",
                                    "CLEC5A","CD83","TGFBI","IL1B", 
                                    "MMP9","ACP5", "CST3", "ITGAV","CTSK", "CKB"), 
                       group.by="ident", scale=T, min.percent = 0.0001, max.percent = 1) + 
  theme(legend.position="right",axis.title.x = element_text(size = 14), 
  axis.title.y = element_text(size = 14),  
  axis.text.x = element_text(size = 12),   
  axis.text.y = element_text(size = 12)    
) +
  scale_y_discrete(limits=rev) + 
  scale_colour_gradient(low = "white", high = "black", name="average\nexpression") +
  theme(legend.position = "right",
          legend.direction = "vertical") +
  guides(color = guide_colorbar(order = 1), 
       size = guide_legend(order = 2, ncol=1), ncol=1)
dp

In [ ]:
ggplotColours <- function(n = 6, h = c(0, 360) + 15){
  if ((diff(h) %% 360) < 1) h[2] <- h[2] - 360/n
  hcl(h = (seq(h[1], h[2], length = n)), c = 100, l = 65)
}

color_list <- ggplotColours(n=9)

In [ ]:
names(color_list) <- levels(combined_obj)

In [ ]:
# custom order for further vizualisations
combined_obj$patient <- factor(combined_obj$patient,
                                 levels=c("p19","p4","p15","p16","p10","p6","p3",
                                          "p5","p9","p13","p17","p14","p7","p8","p2"))

levels(combined_obj) <- c("0","3","2","6_0","6_1","1","4","5","7")

In [ ]:
rel <- dittoBarPlot(combined_obj, "ident", group.by = "patient", color.panel = color_list,
                      retain.factor.levels = TRUE) + 
            labs(title = "Relative proportion of \n clusters cells per patient") + 
            theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
            text =  element_text(hjust = 0.5,size = 14))

abs <- dittoBarPlot(combined_obj, "ident", group.by = "patient", color.panel = color_list,
            scale = "count", retain.factor.levels = TRUE) + 
            labs(title ="Absolute proportion of \n clusters cells per patient") + 
            theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
            text =  element_text(hjust = 0.5, size = 14))+ NoLegend()

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 6, warn=-1,verbose = FALSE)
abs+rel

In [ ]:
genes <- c(
  "CD68", "CD14","SPP1", "PLXDC2", "CD163","MRC1","CD209", "MSR1", "IL18", "SELENOP", "GLUL",
  "C1QC", "C1QB", "C1QA","RNASE1" ,"APOC1", "FCGR3A", "VSIG4", "MARCO", "CD109","FOLR2",
  "HMOX1", "PLIN2", "CLEC5A","CD83","CD300E","VCAN","FCN1", 
  "MMP9","ACP5", "CST3","CTSK", "CKB",
  "CLEC9A", "XCR1","HLA-DRA", "HLA-DQA1", "WDFY4","CD3D", "CD8A", "CD8B",
  "TOX", "PDCD1", "HAVCR2", "GZMK", "CD4", "IL7R", "SELL", "CCR7",
  "TCF7", "FOXP3", "CTLA4", "TIGIT","MS4A1", "PAX5", "CD79A", "CD79B",
  "IGHM", "IGLC2", "IGKC")

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 5)
dp1 <- dotplot_marker(combined_obj, genes, TRUE,"Immune Clusters")
dp1

#### Cluster 0

In [ ]:
c="0"
rna.markers %>% 
    filter(cluster == c) %>% 
    head(n=30)

Stromal M2 profile with overexpression of PLXDC2

https://doi.org/10.3389/fcell.2021.673295

https://doi.org/10.1016/j.devcel.2024.08.003

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10, warn=-1,verbose = FALSE)
stromal_M2 = c("CD68", "CD14", "PLXDC2", "CD163", "CXCL12", "MSR1",
                               "CD36", "CD184", "MS4A4A", "VSIG4", "MEF2C", 
              "FCGR3A", "CLEC5A")
VlnPlot(combined_obj, feature=stromal_M2,ncol=4, pt.size = 0.01)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 15)
FeaturePlot(combined_obj, reduction = "umap.rna.harmony",
            features = c("CLEC5A","CD83","CD300E","VCAN","FCN1","AREG","LYZ","MRC1","CD163"), ncol=3, order=T)

#### Cluster 1

T-cells with SKAP1 high, proxy of activated?

https://www.nature.com/articles/s41598-019-45627-9 

THEMIS? Themocyte??

https://pmc.ncbi.nlm.nih.gov/articles/PMC2848698/ 

In [ ]:
c="1"
rna.markers %>% 
    filter(cluster == c) %>% 
    head(n=30)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 15, warn=-1,verbose = FALSE)
markers = c("CD3E", "CD3D", "CD8A", "CD8B","CD2", "CD247","SKAP1","THEMIS", "CD96", "CCL5", "GZMA", "IL7R")
VlnPlot(combined_obj, feature=markers, ncol=3, pt.size = 0.01)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10)
naive <- c("CD4","IL7R", "SELL","CCR7", "TCF7")
VlnPlot(combined_obj, feature=naive, ncol=3, pt.size = 0.01)

In [ ]:
FeaturePlot(combined_obj, feature=c("TOX", "ENTPD1", "LAYN","CTLA4", "PRDM1","PDCD1", 
                                    "LAG3", "TIGIT", "HAVCR2", "GZMK", "FOXP3"), order=T)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 15)
FeaturePlot(combined_obj, feature=c("IL7R", "CCR7", "TCF7","KLRG1", "PRF1", "CX3CR1"), order=T)

#### Cluster 2

In [ ]:
c="2"
rna.markers %>% 
    filter(cluster == c) %>% 
    head(n=30)

In [ ]:
proinf <- c("C1QA","C1QC","CCL2","IL1B","CCL4","CCL7","CCL8","NFKB","CD40", "CXCL2", "CXCL3", 
  "CXCL9", "CXCL10","CXCL11","IDO1","NFKBIA", "TNF","CXCL8","G0S2","IL6","INHBA","S100A8","S100A9")

antinf <- c("SELENOP", "MRC1", "CCL18","CD163", "CD209", "ARG1", "IL10", "CD274","CHIT1",  "RNASE1", 
            "TREM2", "IL10", "ITGA4", "LGALS9", "MARCO", "TGFB2", "TGFB1", "CSFR", "CSF", "SPP1","TREM2")

In [ ]:
options(repr.plot.width = 22, repr.plot.height = 15, warn=-1,verbose = FALSE)
VlnPlot(combined_obj, feature=proinf, idents=c("0","3","2"), ncol=5, pt.size=0.01)

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 15, warn=-1,verbose = FALSE)
VlnPlot(combined_obj, feature=antinf, idents=c("0","3","2"), ncol=5, pt.size=0.01)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 20, warn=-1,verbose = FALSE)
markers = c("CD68", "CD14", "PLXDC2", "CD163","VSIG4","SPP1", "C1QC", "C1QB","C1QA",
            "APOC1", "TYROBP", "HMOX1", "TOP2A", "MKI67")
VlnPlot(combined_obj, feature=markers, ncol=3, pt.size=0.01)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 15, warn=-1,verbose = FALSE)
markers = c("CD68", "CD14", "PLXDC2", "CD163","CD109","HMOX1", "MARCO", "PLIN2", 
            "CD36", "CXCL8", "FN1", "RNASE1")
VlnPlot(combined_obj, feature=markers, ncol=3, pt.size=0.01)

#### Cluster 4

Plasma Cells

In [ ]:
c="4"
rna.markers %>% 
    filter(cluster == c) %>% 
    head(n=30)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, warn=-1,verbose = FALSE)
markers = c("CD3D","CD4","RORA","IL7R","FOXP3","CTLA4")

FeaturePlot(combined_obj, feature=markers, ncol=3, pt.size=0.01, order=T)

#### Cluster 5

mixture Tregs

In [ ]:
c="5"
rna.markers %>% 
    filter(cluster == c) %>% 
    head(n=30)

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 6, warn=-1,verbose = FALSE)
VlnPlot(combined_obj, feature=c("IGKC", "IGHM"),  ncol=2, pt.size=0.01)

#### Cluster 6_0

Dendritic cells

In [ ]:
c="6_0"
rna.markers %>% 
    filter(cluster == c) %>% 
    head(n=30)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 15, warn=-1,verbose = FALSE)
markers = c("CLEC9A","CCSER1", "HLA-DQA1", "PLAC8", "MS4A4E", "WDFY4", 
            "HLA-DQB1", "IDO1", "IRF8", "THBD")

VlnPlot(combined_obj, feature=markers,ncol=3, pt.size=0.01)

#### Cluster 6_1

Osteoclast cells

In [ ]:
c="6_1"
rna.markers %>% 
    filter(cluster == c) %>% 
    head(n=30)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, warn=-1,verbose = FALSE)
osteo_markers = c("MMP9","ACP5", "CST3","CTSK", "CKB")

VlnPlot(combined_obj, feature=osteo_markers, ncol=3, pt.size=0.01)

#### Cluster 7

Bcells

In [ ]:
c="7"
rna.markers %>% 
    filter(cluster == c) %>% 
    head(n=30)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 15, warn=-1,verbose = FALSE)
bcell_markers = c("CD19","MS4A1","CCR7","SELL","ITGAX","CD79A","CD79B", "JCHAIN","IGHM", "BANK1")

VlnPlot(combined_obj, feature=bcell_markers, ncol=3, pt.size=0.01)

In [ ]:
write_csv(rna.markers, here(glue("{int}/{robj_dir}/4.2.1-immune_fraction_markers.csv")))

### Rename

In [ ]:
combined_obj$immune_l2_clusters <- Idents(combined_obj)

In [ ]:
rel <- dittoBarPlot(combined_obj,"patient" , group.by = "immune_l2_clusters", 
                     color.panel = colors_patients, retain.factor.levels = TRUE) + 
        labs(title = "Relative proportion of \n patient cells per cluster") +
        theme(plot.title = element_text(hjust = 0.5),
             text = element_text(size=16)) + NoLegend()
abs <- dittoBarPlot(combined_obj, "patient", group.by = "immune_l2_clusters", 
             scale = "count",color.panel = colors_patients, retain.factor.levels = TRUE) + 
            labs(title ="Absolute proportion of \n patient cells per cluster") + 
            theme(plot.title = element_text(hjust = 0.5),
                 text = element_text(size=16))

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 6, warn=-1,verbose = FALSE)
rel+abs

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 6.5)
DimPlot(combined_obj, 
        reduction = "umap.rna.harmony",
        group.by="patient",
        label = F,
        label.box = TRUE,
        label.size = 4.5,
        repel = TRUE,
        cols = unlist(colors_patients)) + 
    ggtitle("UMAP patient") +
    theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"))


In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20)
DimPlot(combined_obj, 
        reduction = "umap.rna.harmony",
        group.by="patient",
        split.by="patient",
        ncol=5,
        label = F,
        label.box = TRUE,
        label.size = 4.5,
        repel = TRUE,
        cols = unlist(colors_patients)) 

In [ ]:
rename_clusters <- c("0"="c0-3:Antiinflam. Mφ",
                     "3"="c0-3:Antiinflam. Mφ",
                     "2"="c2:MARCO TAM", 
                     "6_1"="c6_1:Osteoclast",
                     "6_0"="c6_0:Dendritic cells",
                     "1"="c1:CD8 pre-exhausted", 
                     "5"="c5:Treg", 
                     "7"="c7:B cells",
                     "4"="c4:Plasma cells")
combined_obj <- RenameIdents(object = combined_obj, rename_clusters)
combined_obj$immune_l2_clusters <- Idents(combined_obj)

In [ ]:
color_immune = list(
  "c0-3:Antiinflam. Mφ" = "#c83771ff",
  "c2:MARCO TAM" = "#f4d7d7ff",
  "c6_0:Dendritic cells" = "#5a1a7aff",
  "c6_1:Osteoclast" = "#cfa6f3ff",
  "c1:CD8 pre-exhausted" = "#e31a1c",
  "c5:Treg" = "#800000ff",     
  "c4:Plasma cells" = "#ff4f1aff",  
  "c7:B cells" = "#ff9a76ff"
)

In [ ]:
generate_color <- function(names_list) {
  colors <- ifelse(names_list %in% c("c0-3:Antiinflam. Mφ", 
                                     "c1:CD8 pre-exhausted",
                                     "c5:Treg", 
                                     "c6_0:Dendritic cells"), "white", "black")
  return(colors)
}

color_labels <- generate_color(unique(Idents(combined_obj)))

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 6)
p_clust <- DimPlot(combined_obj, 
        reduction = "umap.rna.harmony",
        group.by="immune_l2_clusters",
        label = T,
        label.box = TRUE,
        label.color = as.list(color_labels),
        label.size = 4.5,
        repel = TRUE,
        cols = unlist(color_immune)) + 
    ggtitle("Immune Clusters") + NoLegend() +
    theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"))

p_clust

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 5)
dp1 <- dotplot_marker(combined_obj, genes, TRUE,"Immune Clusters")
dp1

In [ ]:
rel <- dittoBarPlot(combined_obj, "ident", group.by = "patient", color.panel = color_immune,
                      retain.factor.levels = TRUE ) + 
            labs(title = "Relative proportion of \n clusters cells per patient") + 
            theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
            text =  element_text(hjust = 0.5,size = 14, color = "black"))

abs <- dittoBarPlot(combined_obj, "ident", group.by = "patient", color.panel = color_immune,
            scale = "count", retain.factor.levels = TRUE) + 
            labs(title ="Absolute proportion of \n clusters cells per patient") + 
            theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
            text =  element_text(hjust = 0.5, size = 14,color = "black"))+ NoLegend()

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 6, warn=-1,verbose = FALSE)
bp <- abs+rel
bp

## Save

In [ ]:
write_csv(rna.markers, here(glue("{int}/{robj_dir}/4.2.1-immune_fraction_markers.csv")))

In [ ]:
source(here::here("utils/seurat2shiny.R"))
seurat2shiny(combined_obj, tech = "sc", reduction="umap.rna.harmony",
             save = T, path = here::here(glue::glue("{int}/{robj_dir}/")))

In [ ]:
ggsave(plot = p_clust, width = 7, height = 6, dpi = 300, 
       file = here::here(glue::glue("{int}/{plt_dir}/4.2.1-immune_fraction_UMAP.svg")))

In [ ]:
ggsave(plot = dp1, width = 18, height = 5, dpi = 300, 
       file = here::here(glue::glue("{int}/{plt_dir}/4.2.1-immune_fraction_dotplot.svg")))

In [ ]:
ggsave(plot = bp, width = 12, height = 6, dpi = 300, 
       file = here::here(glue::glue("{int}/{plt_dir}/4.2.1-immune_fraction_barplot.pdf")))

In [ ]:
saveRDS(combined_obj, here(glue("{int}/{robj_dir}/4.2.1-immune_fraction.rds")))

In [ ]:
sessionInfo()